# Fetal ECG: train and compare machine-learning models

This notebook trains **9 standard classifiers** on the same task and compares them fairly, then saves
them so the Streamlit app can use and display them.

**The task:** for every candidate peak found in the abdominal ECG, decide *"is this a fetal heartbeat?"*.
The direct fetal ECG (`Direct_1`) provides the correct answers for training and scoring.

**How the comparison is fair:** all models get identical data and features. Each recording is held out
in turn (*leave-one-recording-out*), and the confidence threshold is chosen using only the training
recordings, so no model is ever scored on data it learned from.

**Before you start:** put the dataset CSV files (one per recording, with `Abdomen_1..4` and `Direct_1`)
in a folder, and set `DATA_DIR` in the *Data* cell below. In Colab: upload them to the Files panel
or mount Google Drive.

In [ ]:
# 1. Setup - works in Colab or locally. If the repo code is not found next to this notebook it is cloned.
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/YOUR-USERNAME/YOUR-REPO.git"   # <- edit this (only needed in Colab)

def find_repo():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/repo")]:
        if (p / "fecg").is_dir():
            return p
    return None

REPO = find_repo()
if REPO is None:
    subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    REPO = Path("/content/repo")
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from fecg import evaluation as ev, models as fm, pipeline as fp
try:
    from IPython.display import display
except ImportError:
    display = print

print("Repo:", REPO, "| scikit-learn", sklearn.__version__)

In [ ]:
# 2. Data - folder containing one CSV per recording (e.g. r01.csv, r04.csv, ...)
# To read from Google Drive instead, run:  from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = Path(os.environ.get("FECG_DATA", "/content/data"))
FS = 1000          # sampling rate in Hz

csv_files = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in {DATA_DIR}")
assert len(csv_files) >= 3, "Need at least 3 recordings - check DATA_DIR."

In [ ]:
# 3. Run the signal pipeline on every recording (filter -> remove maternal ECG -> candidate peaks -> features)
procs = {}
for f in csv_files:
    try:
        proc = fp.process_record(f.read_bytes(), FS)
    except Exception as e:
        print(f"skipped {f.name}: {e}")
        continue
    if proc["ref"] is None:
        print(f"skipped {f.name}: no Direct_1 column")
        continue
    procs[f.stem] = proc
    print(f"loaded {f.name}: {proc['n'] / FS:.1f} s")

cs = ev.build_candidate_set(procs)
print(f"\n{len(cs.y)} candidate peaks from {len(procs)} recordings: {int(cs.y.sum())} are fetal beats, "
      f"{int((cs.y == 0).sum())} are not.")

In [ ]:
# 4. Train and evaluate all models (leave-one-recording-out). This takes a few minutes.
zoo = fm.build_zoo()
result = ev.benchmark_models(cs, zoo, progress=print)
summary = result["summary"]

In [ ]:
# 5. Results table (means over the held-out recordings)
cols = ["Model", "Type", "Sensitivity", "Precision", "F1", "F1_std", "AUC", "FHR_err_BPM",
        "FHR_MAE_BPM", "Train_s", "Infer_ms_per_1000"]
display(summary[cols].round(3))
print("Sensitivity = share of true fetal beats found. Precision = share of detections that are real.")
print("F1 = balance of the two. AUC = how well scores separate fetal from non-fetal peaks (candidate level).")
print("FHR_err_BPM = error of the median heart rate. Train_s / Infer_ms = efficiency.")

In [ ]:
# 6. Charts
s = summary.iloc[::-1].reset_index(drop=True)           # best on top
fig, ax = plt.subplots(figsize=(10, 0.6 * len(s) + 1.8))
y = np.arange(len(s)); h = 0.26
for i, (col, c) in enumerate((("Sensitivity", "#0f9d9a"), ("Precision", "#e0457b"), ("F1", "#0b3b4a"))):
    ax.barh(y + (i - 1) * h, s[col], height=h, color=c, label=col)
ax.set_yticks(y); ax.set_yticklabels(s["Model"]); ax.set_xlim(0, 1.05)
ax.legend(loc="lower right"); ax.grid(axis="x", alpha=.3)
ax.set_title("Beat detection quality (leave-one-recording-out)")
plt.tight_layout(); plt.show()

from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(6, 5))
for key, p in result["oof"].items():
    fpr, tpr, _ = roc_curve(cs.y, p)
    ax.plot(fpr, tpr, label=f"{zoo[key][0]} (AUC {summary.set_index('Key').loc[key, 'AUC']:.2f})")
ax.plot([0, 1], [0, 1], "k:", lw=1)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curves (candidate level, out-of-fold)"); ax.legend(fontsize=7); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

ml = summary[summary["Type"] == "ml"].sort_values("Train_s")
fig, ax = plt.subplots(1, 2, figsize=(12, 0.45 * len(ml) + 1.5))
ax[0].barh(ml["Model"], ml["Train_s"], color="#0f9d9a"); ax[0].set_title("Training time (s)")
ax[1].barh(ml["Model"], ml["Infer_ms_per_1000"], color="#e0457b"); ax[1].set_title("Inference (ms per 1000 candidates)")
for a in ax: a.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# 7. How much do results vary between recordings? (F1 per held-out recording)
pr = result["per_record"]
display(pr.pivot_table(index="Model", columns="Record", values="F1").round(3))
print("With only a few short recordings, models whose F1 differ by less than the F1_std spread (see the table above) are effectively tied.")

In [ ]:
# 8. Save the other models, the benchmark tables and a pinned requirements.txt into the repo's models/ folder,
#    then package them for download. Upload the contents of the zip to your GitHub repository.
SKIP_SAVE = ["random_forest"]   # keep the Random Forest you already trained (fecg_model.joblib); use [] to replace it
ev.save_run(REPO / "models", result, cs.fs, cs.rec_ids, zoo, skip=SKIP_SAVE)
print("Pinned:", fm.pin_requirements(REPO / "requirements.txt"))

UPD = Path("fecg_models_update")
shutil.rmtree(UPD, ignore_errors=True)
shutil.copytree(REPO / "models", UPD / "models")
shutil.copy(REPO / "requirements.txt", UPD / "requirements.txt")
archive = shutil.make_archive("fecg_models_update", "zip", root_dir=UPD)
print("Created", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("(Not in Colab - the zip is in the current folder.)")